# 02 - GNN Train

Run hybrid Neo4j + Postgres GraphSAGE training and persist outputs to AI tables.

In [7]:
import json
import sys
from sqlalchemy import text

if '/app' not in sys.path:
    sys.path.append('/app')

from app.ledger.db import SessionLocal
from app.core.config import settings
from app.analytics.layer3.gnn_train_worker import run_once as run_gnn_train

In [8]:
db = SessionLocal()
try:
    result = run_gnn_train(
        db=db,
        window_key=settings.gnn_window_key,
        edge_backend=settings.gnn_edge_backend,
        max_entities=settings.gnn_max_entities,
        max_edges=settings.gnn_max_edges,
        min_edge_weight=settings.gnn_min_edge_weight,
        epochs=settings.gnn_epochs,
        hidden_dim=settings.gnn_hidden_dim,
        embed_dim=settings.gnn_embed_dim,
        dropout=settings.gnn_dropout,
        learning_rate=settings.gnn_learning_rate,
        weight_decay=settings.gnn_weight_decay,
        seed=settings.gnn_seed,
        model_version=settings.gnn_model_version,
        prediction_type=settings.gnn_prediction_type,
        artifact_dir=settings.gnn_artifact_dir,
    )
    print(json.dumps(result, indent=2))
finally:
    db.close()

{
  "status": "ok",
  "gnn_run_id": "a0600327-fced-4f82-8642-1053e3a840b4",
  "window_key": "Wmid",
  "window_end": "2026-01-02T14:33:20.916951+00:00",
  "nodes": 37,
  "edges": 65,
  "source_backend": "postgres",
  "embeddings_upserted": 37,
  "predictions_created": 37,
  "predictions_updated": 0,
  "metrics": {
    "accuracy": 0.783784,
    "precision": 1.0,
    "recall": 0.771429,
    "f1": 0.870968,
    "auc": 0.985714,
    "train_loss": 0.031467,
    "val_loss": 0.007466
  },
  "artifact_path": "/app/artifacts/gnn/a0600327-fced-4f82-8642-1053e3a840b4.pt"
}


In [9]:
db = SessionLocal()
try:
    q = text('''
        SELECT id, model_version, prediction_type, auc, precision, recall, f1, created_at
        FROM gnn_training_run
        ORDER BY created_at DESC
        LIMIT 5
    ''')
    for row in db.execute(q).fetchall():
        print(row)
finally:
    db.close()

(UUID('a0600327-fced-4f82-8642-1053e3a840b4'), 'gnn-sage-v1', 'risk_gnn', 0.985714, 1.0, 0.771429, 0.870968, datetime.datetime(2026, 2, 15, 5, 53, 46, 176697, tzinfo=datetime.timezone.utc))
